In [ ]:
!pip install autogluon.timeseries

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.9/827.9 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

In [ ]:
url = "https://autogluon.s3.amazonaws.com/datasets/timeseries/m4_hourly/train.csv"
df = pd.read_csv(url)

print("Форма данных:", df.shape)
df.head()

Форма данных: (353500, 3)


,item_id,timestamp,target
0,H1,1750-01-01 00:00:00,605.0
1,H1,1750-01-01 01:00:00,586.0
2,H1,1750-01-01 02:00:00,586.0
3,H1,1750-01-01 03:00:00,559.0
4,H1,1750-01-01 04:00:00,511.0


In [ ]:
prediction_length = 48
series_id = 'H1'

series_data = df[df['item_id'] == series_id]['target'].values
train_data = series_data[:-prediction_length]
test_data = series_data[-prediction_length:]

print(f"Train: {len(train_data)} часов")
print(f"Test: {len(test_data)} часов")

Train: 652 часов
Test: 48 часов


In [ ]:
train_timestamps = pd.date_range(
    start="2000-01-01 00:00:00",
    periods=len(train_data),
    freq="H"
)

train_df = pd.DataFrame({
    "item_id": series_id,
    "timestamp": train_timestamps,
    "target": train_data
})

print("Train data for AutoGluon:")
print(train_df.head())

Train data for AutoGluon:
  item_id           timestamp  target
0      H1 2000-01-01 00:00:00   605.0
1      H1 2000-01-01 01:00:00   586.0
2      H1 2000-01-01 02:00:00   586.0
3      H1 2000-01-01 03:00:00   559.0
4      H1 2000-01-01 04:00:00   511.0


/tmp/ipykernel_14942/1073242650.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  train_timestamps = pd.date_range(


In [ ]:
train_data_ts = TimeSeriesDataFrame(train_df)

print(f"Shape: {train_data_ts.shape}")
print(train_data_ts.head())

Shape: (652, 1)
                             target
item_id timestamp                  
H1      2000-01-01 00:00:00   605.0
        2000-01-01 01:00:00   586.0
        2000-01-01 02:00:00   586.0
        2000-01-01 03:00:00   559.0
        2000-01-01 04:00:00   511.0


In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    eval_metric="MASE",
    target="target"
)

predictor.fit(
    train_data_ts,
    presets="medium_quality",
    time_limit=300
)

Beginning AutoGluon training... Time limit = 300s
AutoGluon will save models to '/content/AutogluonModels/ag-20260515_193241'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       11.37 GB / 12.67 GB (89.7%)
Disk Space Avail:   75.97 GB / 107.72 GB (70.5%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 48,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': Fals

In [ ]:
print("\n" + "="*50)
print("Leaderboard - Сравнение всех моделей")
print("="*50)
leaderboard = predictor.leaderboard()
print(leaderboard)


Leaderboard - Сравнение всех моделей
              model  score_val  pred_time_val  fit_time_marginal  fit_order
0  WeightedEnsemble  -0.493211       0.183488           0.179118          6
1     SeasonalNaive  -0.502059       0.009980           0.026657          1
2             Theta  -0.657433       0.055451           0.019999          5
3               ETS  -0.710401       2.992458           0.013496          4
4     DirectTabular  -0.957510       0.088939           0.672348          3
5  RecursiveTabular  -0.957886       0.171603           0.805450          2
